# 10 - Trainer Overfitting a Single Batch

This notebook exercises the full `Trainer` end to end on CPU using the
smallest backbone and a single tiny synthetic batch, with the
`OverfitManager` enabled so the same batch is reused every step. A correctly
wired trainer (model forward, loss, backward, optimizer step, scheduler,
checkpoint, early stopping) must drive the training loss steadily toward
zero. We plot the loss curve and compare the predicted profile to the target
before and after training.

Modules exercised: `pipelines.training_pipeline.trainer.Trainer`,
`TrainStep`, `OverfitManager`, plus the model registry and `Loss`.

Note: running this notebook trains for a handful of epochs on CPU; it is
intentionally tiny (batch size 2, two-level UNet, small patches).

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy             as np
import torch
import matplotlib.pyplot as plt

SEED   = 0
DEVICE = torch.device("cpu")

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)

plt.rcParams.update({
    "figure.dpi"     : 120,
    "savefig.dpi"    : 300,
    "font.size"      : 10,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
})

print("repo root :", REPO_ROOT)
print("torch     :", torch.__version__)
print("device    :", DEVICE)


In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = ""
torch.cuda.is_available = lambda: False
assert not torch.cuda.is_available(), "the Trainer must see CPU only in this notebook"

from configuration.training_config import (
    TrainerConfig, GaussianConfig, TrainingConfigInner, WarmupConfig,
    EMAConfig, OverfitConfig, ResourceConfig, MemoryConfig,
    LossConfig, LossCurriculumConfig,
)
from configuration.models_config import UNetConfig
from models import get_model
from pipelines.training_pipeline.trainer import Trainer
from tools.logger import Logger

class IdentityNormStats:
    def normalize_output(self, tensor):
        return tensor

    def denormalize_output(self, tensor):
        return tensor

X_MIN, X_MAX = -10.0, 40.0
x_axis       = np.linspace(X_MIN, X_MAX, 16, dtype=np.float32)
gaussian_cfg = GaussianConfig(n_default_gaussians=2, x_min=X_MIN, x_max=X_MAX)


## Assemble a CPU trainer config

Everything is forced small and onto CPU. The `OverfitManager` collapses the
loader to a single batch repeated `max_steps` times. EMA and the resource
monitor are disabled to keep the run light. The loss combines a curve term
and a parameter term.

In [ ]:
warmup_loss = LossConfig(use_param_l1=True, weight_param_l1=1.0,
                         use_mse_curve=True, weight_mse_curve=1.0)

cfg = TrainerConfig(gaussian=gaussian_cfg)
cfg.curriculum = LossCurriculumConfig(enabled=False, warmup=warmup_loss, complete=warmup_loss)
cfg.training   = TrainingConfigInner(device='cpu', epochs=300, validation_frequency=999,
                                     use_amp=False, log_debug=False, log_all_losses=False)
cfg.warmup     = WarmupConfig(warmup_steps=15, warmup_enabled=True, warmup_mode='linear')
cfg.ema        = EMAConfig(use_ema=False)
cfg.resources  = ResourceConfig(enabled=False, log_to_csv=False, log_to_tensorboard=False)
cfg.memory     = MemoryConfig(clear_cache_after_epoch=False, clear_cache_after_eval=False)
cfg.overfit    = OverfitConfig(enabled=True, max_steps=300, stop_threshold=1e-8, batch_size=2)
cfg.scheduler.type   = 'constant'
cfg.io.writer        = None

model_cfg = UNetConfig(in_channels=4, out_channels=6, features=[16, 32],
                       bottleneck_factor=1, dropout=0.0, normalization='instance',
                       encoder_lr=8e-3, bottleneck_lr=8e-3, decoder_lr=8e-3, output_head_lr=1.2e-2,
                       encoder_wd=0.0, bottleneck_wd=0.0, decoder_wd=0.0, output_head_wd=0.0)
model, mc = get_model('unet', config=model_cfg)
print('trainable params:', sum(p.numel() for p in model.parameters() if p.requires_grad))


## Build one synthetic batch with a known target

The input is random; the target is a fixed, physically plausible pair of
Gaussians shared across the batch. The network must memorise the mapping from
this specific input to the target parameters.

In [ ]:
torch.manual_seed(SEED)
B, C_in, H, W = 2, 4, 8, 8
images = torch.randn(B, C_in, H, W)

gt = torch.zeros(B, 6, H, W)
gt[:, 0] = 1.0; gt[:, 1] =  5.0; gt[:, 2] = 3.0
gt[:, 3] = 0.6; gt[:, 4] = -3.0; gt[:, 5] = 2.0

train_loader = [(images, gt)]
logger = Logger(log_dir='/tmp/nb10_logs', name='overfit', level='WARNING')

trainer = Trainer(model, mc, x_axis, cfg, run_dir='/tmp/nb10_run',
                  logger=logger, norm_stats=IdentityNormStats(), emit_docs=False)
print('trainer device:', trainer.device)


In [ ]:
criterion = trainer.criterion
x_axis_t  = torch.tensor(x_axis)
images_d  = images.to(trainer.device)
gt_d      = gt.to(trainer.device)

with torch.no_grad():
    pred_before  = trainer.model(images_d)
    curve_before = criterion.reconstruct_gaussians(pred_before.float()).mean(dim=(0, 2, 3)).cpu().clone()

train_losses, val_losses, best = trainer.train(train_loader, train_loader, train_loader)

with torch.no_grad():
    pred_after  = trainer.model(images_d)
    curve_after = criterion.reconstruct_gaussians(pred_after.float()).mean(dim=(0, 2, 3)).cpu().clone()
    curve_gt    = criterion.reconstruct_gaussians(gt_d.float()).mean(dim=(0, 2, 3)).cpu().clone()

print('first train loss : %.5f' % train_losses[0])
print('last  train loss : %.5f' % train_losses[-1])


## Loss decay and profile recovery

The left panel shows the training loss per epoch on a log scale; the right
panel overlays the predicted elevation profile before and after training
against the target.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(np.arange(1, len(train_losses) + 1), train_losses, marker='.', color='C0')
axes[0].set_yscale('log')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('train loss (log)')
axes[0].set_title('Overfit loss decay')

xs = x_axis_t.numpy()
axes[1].plot(xs, curve_gt.numpy(),     color='black', lw=2, label='target')
axes[1].plot(xs, curve_before.numpy(), color='C3', ls='--', label='before training')
axes[1].plot(xs, curve_after.numpy(),  color='C0',          label='after training')
axes[1].set_xlabel('elevation x [m]'); axes[1].set_ylabel('reflectivity [a.u.]')
axes[1].set_title('Predicted profile vs target')
axes[1].legend(frameon=False)
fig.tight_layout()
plt.show()


## Expected visual outcome

The training loss should fall by more than an order of magnitude over the run
as the tiny network fits the single batch, then flatten onto a small residual.
The predicted elevation profile (averaged over the patch) should start far from
the target (red dashed) and converge closely onto it (blue over black) by the
end of training, confirming the full trainer step, loss, backward pass, and
optimisation loop are correctly connected. The curve term supplies the gradient
that recovers the Gaussian widths, which the parameter term alone cannot drive
through the width clamp.